In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install flowio

In [ ]:
import pandas as pd
import os
import sys
import pickle
import seaborn as sns

In [ ]:

fixed_path = '/content/drive/MyDrive/0.Master_Thesis/'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)

cellcnn_path = f'{fixed_path}CellCNN/'
if cellcnn_path not in sys.path:
    sys.path.append(cellcnn_path)

save_path = f'{cellcnn_path}results/'
if save_path not in sys.path:
    sys.path.append(save_path)

modules_dir = f'{cellcnn_path}modules/'
if modules_dir not in sys.path:
    sys.path.append(modules_dir)

In [ ]:
decache_files = ['utils', 'timepoints_elaboration', 'cv_folds', 'show_results']

# Remove from cache
from utils import remove_from_cache
remove_from_cache(decache_files)


from timepoints_elaboration import load_data
from utils import show_blast_distribution_perc
from cv_folds import generate_LOPOCV_dicts, generate_LOPOCV_folds

from show_results import retrieve_samples_info, show_patients_samples_info, load_tuning_data
from show_results import  generate_dict_comb_3d, show_heat_map_combinations, retrieve_all_LOPO_ensemble_thresholds

# Show BO tuning

In [ ]:

data_folder_dir = f'{fixed_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

multiple_donations, ALL_DATASETS = load_data(data_path = data_folder_dir, ext = extension)#, remove_control = True)


In [ ]:
tuning_exp = 'Trial_4_NO_AS_bayesian_tuning'

config_save_dir = f'{cellcnn_path}/experiments/experiment_{tuning_exp}/'

with open(os.path.join(config_save_dir, 'config.pkl'), 'rb') as f:
            config = pickle.load(f)

starting_seed = config['starting_seed']

plot_exp = '4_plots'
thesis_images_dir = f'{cellcnn_path}/experiments/experiment_{plot_exp}/thesis_images'
os.makedirs(thesis_images_dir, exist_ok=True)

In [ ]:

full_LOPOCV_dicts = generate_LOPOCV_dicts(multiple_donations, ALL_DATASETS)
LOPOCV_patients_folds = generate_LOPOCV_folds(full_LOPOCV_dicts, ALL_DATASETS, starting_seed)

In [ ]:

save_lopo_ens_dir = f'{cellcnn_path}/experiments/experiment_{tuning_exp}/'
LOPO_folds = len([fold_name for fold_name in list(os.listdir(save_lopo_ens_dir)) if 'fold' in fold_name])
print(LOPO_folds)


save_tuning_plot_dir = f'{cellcnn_path}/experiments/experiment_{plot_exp}/bayesian_tuning/'
os.makedirs(save_tuning_plot_dir, exist_ok=True)

tuning_exp_dir = f'{cellcnn_path}/experiments/experiment_{tuning_exp}/'

heatmap_dict_3d = generate_dict_comb_3d(LOPO_folds, tuning_exp_dir)
sns.set_style('darkgrid')
show_heat_map_combinations(heatmap_dict_3d, save_dir = save_tuning_plot_dir) # heatmap of the tuned parameters combinations
sns.set_style("ticks")


In [ ]:
tuned_theta_df = pd.DataFrame(heatmap_dict_3d)
tuned_theta_df

# Thresholds from BO and Ensemble post-training

In [ ]:
thrs_dict = {}

BO_roc_thresholds = []
BO_res_thresholds = []
for LOPO_idx, _ in enumerate(range(LOPO_folds)):

    tuning_load_dir = f'{cellcnn_path}/experiments/experiment_{tuning_exp}/outer_fold_{LOPO_idx}/tuning/results'

    threshold_data = load_tuning_data(tuning_load_dir)

    robust_threshold =threshold_data['robust_threshold']
    roc_threshold =threshold_data['roc_threshold']
    BO_roc_thresholds.append(roc_threshold)
    BO_res_thresholds.append(robust_threshold)

thrs_dict['BO_ROC'] = BO_roc_thresholds
thrs_dict['BO_RES'] = BO_res_thresholds

exp = 'Trial_4_NO_AS_Ensemble'
no_as_ens_roc_thr_per_fold, no_as_ens_res_thr_per_fold = retrieve_all_LOPO_ensemble_thresholds(LOPO_folds, cellcnn_path, exp)
thrs_dict['NO_AS_ROC'] = no_as_ens_roc_thr_per_fold
thrs_dict['NO_AS_RES'] = no_as_ens_res_thr_per_fold

exp = 'Trial_4_AS_Ensemble'

as_ens_roc_thr_per_fold, as_ens_res_thr_per_fold = retrieve_all_LOPO_ensemble_thresholds(LOPO_folds, cellcnn_path, exp)
thrs_dict['AS_ROC'] = as_ens_roc_thr_per_fold
thrs_dict['AS_RES'] = as_ens_res_thr_per_fold

thrs_df = pd.DataFrame(thrs_dict)
thrs_df